# Oppgave 5

Song metoden på den fouriertransformerte Cahn-Hilliard ligningen:

$$
\hat{U}^{(1)} = \hat{U}^n + \tau \left[ -(\kappa \left| k \right|^4 + \alpha \left| k \right|^2) \hat{U}^{(1)} + \widehat{\mathcal{N}(U^n)} \right]
$$

$$
\hat{U}^{(2)} = \alpha_{10} \hat{U}^n + \alpha_{11} \hat{U}^{(1)} + \beta_1 \tau \left[ -(\kappa \left| k \right|^4 + \alpha \left| k \right|^2) \hat{U}^{(2)} + \widehat{\mathcal{N}(U^{(1)})} \right]
$$

$$
\hat{U}^{(n+1)} = \alpha_{20} \hat{U}^n + \alpha_{21} \hat{U}^{(1)} + \alpha_{22} \hat{U}^{(2)} + \beta_2 \tau \left[ -(\kappa \left| k \right|^4 + \alpha \left| k \right|^2) \hat{U}^{(n+1)} + \widehat{\mathcal{N}(U^{(2)})} \right]
$$

Hvor $\mathcal{N}(U) = \Delta U^3 - (1 + a) \Delta U$. Og når man løser for de ukjente blir det:

$$
\hat{U}^{(1)} = \frac{\hat{U}^n + \tau \widehat{\mathcal{N}(U^n)}}{(1 + \tau (\kappa \left| k \right|^4 + \alpha \left| k \right|^2)) }
$$
  
$$
\hat{U}^{(2)} = \frac{\alpha_{10} \hat{U}^n + \alpha_{11} \hat{U}^{(1)} + \beta_1 \tau \widehat{\mathcal{N}(U^{(1)})}}{(1 + \beta_1 \tau (\kappa \left| k \right|^4 + \alpha \left| k \right|^2)) }
$$
  
$$
\hat{U}^{(n+1)} = \frac{\alpha_{20} \hat{U}^n + \alpha_{21} \hat{U}^{(1)} + \alpha_{22} \hat{U}^{(2)} + \beta_2 \tau \widehat{\mathcal{N}(U^{(2)})}}{(1 + \beta_2 \tau (\kappa \left| k \right|^4 + \alpha \left| k \right|^2))}
$$



In [4]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fft2, ifft2, fftfreq, fftshift

def cahn_hilliard_backward_euler(*, 
                                kappa, 
                                X, Y, U0, 
                                t0, T, Nt,
                                g, coeff,
                                alpha=1.5):
    """
    Implements the Cahn-Hilliard equation solver using the backward Euler method 
    with a convex-concave splitting approach.

    Parameters:
    -----------
    kappa : float
        Diffusion coefficient for the biharmonic operator.
    X : ndarray
        2D array representing the x-coordinates of the grid.
    Y : ndarray
        2D array representing the y-coordinates of the grid.
    U0 : ndarray
        Initial condition for the solution.
    t0 : float
        Initial time.
    T : float
        Final time.
    Nt : int
        Number of time steps.
    g : callable or None
        Source term as a function of (X, Y, t). If None, no source term is applied.
    alpha : float, optional
        Convex-concave splitting parameter. Default is 1.5.

    Yields:
    -------
    tuple: A tuple containing the discrete Fourier transform of U at t, and the current time t.

    """

    # Prepare relevant data for Fourier transform
    N, M = U0.shape  # U0 is a NxM matrix, but in python .shape gives them in reverse order
    dx = X[0, 1] - X[0, 0]
    dy = Y[1, 0] - Y[0, 0]
    k_x = (fftfreq(N, d=dx/(2*np.pi)))
    k_y = (fftfreq(M, d=dy/(2*np.pi)))
    alpha_10, alpha_11, alpha_20, alpha_21, alpha_22, beta_1, beta_2 = coeff

    KX, KY = np.meshgrid(k_x, k_y, indexing="ij")
    K2 = -(KX**2 + KY**2)
    K4 = K2**2

    # Compute DFT of the initial value
    U_hat = fft2(U0)

    # Add your time-stepping loop here
    # Time stepping 
    t = t0 
    dt = (T-t0)/Nt

    # For convenience when plotting, computing errors, etc., 
    # return the initial solution and initial time.
    yield U_hat, t  

    
    while t < T-dt/2:
        # Compute source term in Fourier space if provided
        if g is not None:
            g_hat = fft2(g(X, Y, t + dt/2))
        else:
            g_hat = 0
        
        def NonLinear(U_hat):
            U = ifft2(U_hat).real
            NonLin = U**3 - (1 + alpha) * U
            return K2*fft2(NonLin)

        # Bruker Song metoden for å finne neste steglengde
        # U_hat er U^n
        U_hat_1 = (U_hat + dt * NonLinear(U_hat) + g_hat) / (1 + dt * (K4 + alpha * K2))
        U_hat_2 = (alpha_10 * U_hat + alpha_11 * U_hat_1 + beta_1 * dt * (NonLinear(U_hat_1)+ g_hat)) / (1 + beta_1 * dt * (K4 + alpha * K2))
        U_hat_np1 = (alpha_20 * U_hat + alpha_21 * U_hat_1 + alpha_22 * U_hat_2 + beta_2 * dt * (NonLinear(U_hat_2) + g_hat)) / (1 + beta_2 * dt * (K4 + alpha * K2))

        # Oppdaterer U_hat og t
        U_hat = U_hat_np1 
        t += dt

        yield U_hat, t

In [5]:
def U_ex(x, y, t):
    return np.sin(x) * np.cos(y) * np.exp(-4*kappa_value*t)

kappa_value = 0

x, y, t, kappa = sp.symbols("x y t kappa")
u_ex = sp.sin(x) * sp.cos(y) * sp.exp(-4*kappa*t)
laplacian_u = sp.diff(u_ex, x, 2) + sp.diff(u_ex, y, 2)
biharmonic_u = sp.diff(laplacian_u, x, 2) + sp.diff(laplacian_u, y, 2)
nonlinear = sp.diff(u_ex**3 - u_ex, x, 2) + sp.diff(u_ex**3 - u_ex, y, 2)
g = sp.diff(u_ex, t) + kappa * biharmonic_u - nonlinear
g_simplified = sp.simplify(g)
sp.pprint(g_simplified)

g_lambdify1 = sp.lambdify((x, y, t), g_simplified.subs(kappa, 1), "numpy")
g_func1 = lambda X, Y, t: g_lambdify1(X, Y, t)
g_lambdify001 = sp.lambdify((x, y, t), g_simplified.subs(kappa, 0.01), "numpy")
g_func001 = lambda X, Y, t: g_lambdify001(X, Y, t)

  ⎛   8⋅κ⋅t        2       2           2           2       ⎞  -12⋅κ⋅t          ↪
2⋅⎝- ℯ      - 9⋅sin (x)⋅sin (y) + 6⋅sin (x) + 3⋅sin (y) - 3⎠⋅ℯ       ⋅sin(x)⋅c ↪

↪      
↪ os(y)


In [6]:
t0, T = 0, 1
Nts = [100, 200, 400, 800, 1600, 3200]

N = 64
x, y = np.linspace(0, 16*np.pi, N, endpoint=False), np.linspace(0, 16*np.pi, N, endpoint=False)
X, Y = np.meshgrid(x, y)
U0 = U_ex(X, Y, 0)

coeff_1 = (3/2, -1/2, 0, 0, 1, 1/2, 1)
coeff_2 = (2, -1, 1/2, 0, 1/2, 1, 1)
coeff_3 = (2, -1, 0, 1/2, 1/2, 1, 1/2)
coeff_4 = (5/2, -3/2, 2/3, 0, 1/3, 3/2, 1)

kappa_value = 1
for coeff in [coeff_1, coeff_2, coeff_3, coeff_4]:
    errors = []
    hs = []
    for Nt in Nts:
        dt = (T - t0) / Nt
        solver = cahn_hilliard_backward_euler(kappa=kappa_value, X=X, Y=Y, U0=U0, t0=t0, T=T, Nt=Nt, g = g_func1, alpha=1.5, coeff = coeff)
        max_error = 0
        for Uhat, t in solver:
            U = ifft2(Uhat).real
            new_error = np.max(np.abs(U - U_ex(X, Y, t)))
            if new_error > max_error:
                max_error = new_error
        
        errors.append(max_error)
        hs.append(dt)

    # Compute EOC
    eocs = []

    for i in range(1, len(errors)):
        eoc = np.log(errors[i] / errors[i-1]) / np.log(hs[i] / hs[i-1])
        eocs.append(eoc)

    # Print results in a table
    print(f"\nConvergence Table for kappa = {kappa_value}:")
    print(f"{'Nt':<10}{'Max Error':<20}{'EOC':<10}")
    print("-" * 40)
    for i, Nt in enumerate(Nts):
        if i == 0:
            print(f"{Nt:<10}{errors[i]:<20.2e}{'-':<10}")
        else:
            print(f"{Nt:<10}{errors[i]:<20.2e}{eocs[i-1]:<10.2f}")

kappa_value = 0.01
for coeff in [coeff_1, coeff_2, coeff_3, coeff_4]:
    errors = []
    hs = []
    for Nt in Nts:
        dt = (T - t0) / Nt
        solver = cahn_hilliard_backward_euler(kappa=kappa_value, X=X, Y=Y, U0=U0, t0=t0, T=T, Nt=Nt, g = g_func001, coeff=coeff, alpha=1.5)
        max_error = 0
        for Uhat, t in solver:
            U = ifft2(Uhat).real
            new_error = np.max(np.abs(U - U_ex(X, Y, t)))
            if new_error > max_error:
                max_error = new_error
        
        errors.append(max_error)
        hs.append(dt)

    # Compute EOC
    eocs = []

    for i in range(1, len(errors)):
        eoc = np.log(errors[i] / errors[i-1]) / np.log(hs[i] / hs[i-1])
        eocs.append(eoc)

    # Print results in a table
    print(f"\nConvergence Table for kappa = {kappa_value}:")
    print(f"{'Nt':<10}{'Max Error':<20}{'EOC':<10}")
    print("-" * 40)
    for i, Nt in enumerate(Nts):
        if i == 0:
            print(f"{Nt:<10}{errors[i]:<20.2e}{'-':<10}")
        else:
            print(f"{Nt:<10}{errors[i]:<20.2e}{eocs[i-1]:<10.2f}")
    

C:\Users\Trond\AppData\Local\Temp\ipykernel_15332\3387922803.py:77: RuntimeWarning: overflow encountered in power
  NonLin = U**3 - (1 + alpha) * U



Convergence Table for kappa = 1:
Nt        Max Error           EOC       
----------------------------------------
100       2.94e+00            -         
200       4.20e+00            -0.52     
400       6.34e+20            -67.03    
800       1.54e+214           -642.41   
1600      5.68e+140           243.94    
3200      8.53e+00            464.48    

Convergence Table for kappa = 1:
Nt        Max Error           EOC       
----------------------------------------
100       2.12e+00            -         
200       3.58e+00            -0.76     
400       1.46e+253           -839.15   
800       2.36e+29            743.41    
1600      2.05e+94            -215.72   
3200      1.10e+276           -603.69   

Convergence Table for kappa = 1:
Nt        Max Error           EOC       
----------------------------------------
100       1.68e+00            -         
200       1.65e+00            0.02      
400       1.56e+00            0.09      
800       1.11e+00            0.48   

KeyboardInterrupt: 